# Metropolis Monte Carlo of Lennard-Jones charged particles

*Utrecht University Molecular Modelling courses from the [Bonvin lab](https://bonvinlab.org).*

This notebook samples a 2D system of Lennard-Jones particles that may also carry a charge
(Coulomb interaction) with the **Metropolis Monte Carlo (MMC)** method. Where the
*energy-minimization* notebooks only ever walk **downhill**, and *molecular dynamics* follows
Newton's equations in time, Monte Carlo takes **random trial moves** and accepts or rejects
them with a rule that makes the simulation sample the **Boltzmann distribution** at a
temperature `T`. Low-energy configurations are visited most often, but uphill moves are
sometimes accepted — so the system can climb out of local traps and explore.

## Theory in brief

### Lennard-Jones
Written from the squared distance $r^2$, with $Z = (r_{min}^2 / r^2)^3 = (r_{min}/r)^6$:

$$E_{LJ} = \varepsilon\, Z (Z-1)$$

The $Z^2$ term is the steep short-range **repulsion** (overlapping atoms), the $-Z$ term the
weaker long-range **attraction**. The two balance at $r = r_{min}$, where the energy reaches
its minimum $-\varepsilon$: `Epsilon` sets the well *depth* and `Rmin` its *position*.

### Coulomb

$$E_{Coul} = \frac{q_a q_b}{\epsilon_r\, r}$$

Like charges repel ($E>0$), unlike charges attract ($E<0$); the dielectric constant
`Dielec` ($\epsilon_r$) screens (weakens) the interaction.

### The Metropolis algorithm
At each step a **trial move** is proposed and its energy change $\Delta E$ is evaluated. The
move is accepted with probability

$$P_{acc} = \min\!\left(1,\; e^{-\Delta E / k_B T}\right):$$

* if the move **lowers** the energy ($\Delta E < 0$) it is *always* accepted;
* if it **raises** the energy it is accepted only with probability $e^{-\Delta E/k_B T}$ — a
  random number $u\in[0,1)$ is drawn and the move is kept if $u < e^{-\Delta E/k_B T}$.

Rejected moves are undone (the system stays where it was, and that configuration is counted
again). This simple rule guarantees, in the long run, that configurations appear with the
Boltzmann weight $e^{-E/k_B T}$ — i.e. Monte Carlo performs **importance sampling** of
thermal equilibrium. Two kinds of trial move are used here:

* a **displacement** — one coordinate of a random atom is shifted by a random amount in
  $[-\texttt{deltaRmax}, +\texttt{deltaRmax}]$ (probability $1-\texttt{frac\_swap}$);
* a **charge swap** — two atoms exchange positions (probability `frac_swap`). Because the
  charges stay attached to their atom index, swapping positions effectively **moves a charge
  across the box in one step**, which helps rearrange the charge pattern without having to
  diffuse through space.

The temperature enters only through $k_B T$: **high** `T` accepts more uphill moves (broad
exploration), **low** `T` accepts almost only downhill moves (behaves like minimization).

## 1. Imports

The numerical core uses only the Python **standard library** (`math`, `random`), so it runs
on a bare Python install. **matplotlib** is the one third-party dependency — it draws the
static figures and the trajectory animation (embedded inline as interactive HTML via
`jshtml`). The cell below first **installs matplotlib if it is missing** (handy on Google
Colab), then imports everything; `%matplotlib inline` renders figures inside the notebook.

In [ ]:
# --- Install required packages if missing (e.g. on Google Colab) ---
import importlib.util, subprocess, sys

for pkg in ["matplotlib"]:
    if importlib.util.find_spec(pkg) is None:
        print(f"Installing {pkg} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
    else:
        print(f"{pkg} already available")

from math import sqrt, exp

import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.animation import FuncAnimation
from matplotlib import rc

from random import random, randint, seed

# Show animations inline
rc('animation', html='jshtml')
%matplotlib inline

## 2. Helper functions

Small utilities used throughout:

* `dist` / `dist2` — Euclidean distance and its square (the squared form avoids a needless
  `sqrt` when we only need to compare distances).
* `SignR(a, b)` — returns `a` with the sign of `b`. It implements the **minimum-image
  convention**: the combination `tmp - SignR(halfbox, tmp-halfbox) - SignR(halfbox, tmp+halfbox)`
  wraps a coordinate difference into the range $[-\tfrac{box}{2}, +\tfrac{box}{2}]$, so each
  atom interacts with the *nearest periodic copy* of its neighbours (periodic boundary
  conditions). The same trick keeps a displaced atom inside the box.
* `charge_color` — purely cosmetic: white for positive charges, dark for negative, used when
  drawing the particles.

In [ ]:
### distance ###
def dist(A, B):
    return sqrt((A[0]-B[0])**2 + (A[1]-B[1])**2)

### squared distance ###
def dist2(A, B):
    return (A[0]-B[0])**2 + (A[1]-B[1])**2

### change sign ###
def SignR(a, b):
    if b > 0:
        return a
    else:
        return -a

### colour particles based on charge ###
def charge_color(charge, qat):
    if charge == qat:
        return "#FFFFFF"   # positive
    else:
        return "#333333"   # negative

## 3. Energy functions

Total energy is the sum over all particle pairs of the Lennard-Jones and Coulomb
contributions, using the **nearest image** convention (periodic boundary conditions).

Distances are handled as **squared** distances (`distsquare`) throughout the inner loops:
this avoids computing a `sqrt` for every pair, and lets the cutoff test (`distsquare <
cutoffsquare`) skip distant pairs cheaply. A `sqrt` is taken only where a term actually
needs $r$ itself (the Coulomb $1/r$). `Calc_Ene2` returns the total energy together with the
separate Lennard-Jones and Coulomb parts, so we can plot their evolution.

In [ ]:
# LJ energy from the squared distance
def LJ2(distsquare, epsilon, rmin_exp6):
    Z = (1/distsquare)**3 * rmin_exp6
    return epsilon * Z * (Z - 1)

# classical Coulomb from the squared distance
def Coulomb2(r, dielec, qa, qb):
    return qa*qb / (dielec*sqrt(r))

# Calculate energy Evdw + Ecoulomb (uses squared distance), with periodic boundary conditions
def Calc_Ene2(coord, epsilon, rmin, dielec, cutoffsquare, boxdim, elec=1):
    Ene = 0.0
    ELJ = 0.0
    ECoul = 0.0
    rmin_exp6 = rmin**6
    # doubly nested loop over all particle pairs
    for i in range(len(coord)-1):
        for j in range(i+1, len(coord)):
            # squared atomic distance (nearest image)
            distsquare = 0
            for k in range(2):
                tmp = coord[j][k] - coord[i][k]
                halfbox = boxdim[k]/2
                tmp = tmp - SignR(halfbox, tmp-halfbox) - SignR(halfbox, tmp+halfbox)
                distsquare += tmp**2
            if distsquare < cutoffsquare:
                qa = coord[i][2]
                qb = coord[j][2]
                vdw = LJ2(distsquare, epsilon, rmin_exp6)
                Ene += vdw
                ELJ += vdw
                if elec:
                    CC = Coulomb2(distsquare, dielec, qa, qb)
                    Ene += CC
                    ECoul += CC
    return Ene, ELJ, ECoul

## 4. The Metropolis Monte Carlo sampler

This is the core of the method — the counterpart of the *minimizer* in the EM notebooks. It
replaces the GUI's `Go` callback / Tkinter event loop with a headless loop over `nsteps` trial
moves. Each step:

1. picks a random atom and, with probability $1-\texttt{frac\_swap}$, proposes a **displacement**
   of one of its coordinates by a random amount in $[-\texttt{deltaRmax},+\texttt{deltaRmax}]$
   (wrapped back into the periodic box); otherwise it proposes a **charge swap** — exchanging
   the positions of two random atoms;
2. evaluates the energy change $\Delta E$ and applies the **Metropolis test**: accept if
   $\Delta E<0$, otherwise accept with probability $e^{-\Delta E/k_B T}$ (a uniform random
   number is drawn only in this uphill case);
3. **undoes** the move if it is rejected, then records the current energy, the running
   acceptance ratio and a snapshot of the configuration.

The running energy and its Lennard-Jones / Coulomb parts are updated **only when a move is
accepted**, so the recorded histories always describe the configuration actually being
visited. (There is deliberately no divide-by-zero floor on `distsquare`: unlike the simplex,
random displacements never land two atoms at *exactly* the same point — a near-overlap simply
costs a huge energy and is rejected.)

In [ ]:
def run_mc(Atom_Coord):
    """Headless Metropolis Monte Carlo. Returns trajectory + energy / acceptance histories."""
    coord = [list(a) for a in Atom_Coord]

    Ene, ELJ, ECoul = Calc_Ene2(coord, Epsilon, Rmin, Dielec, CutOffSquare, BoxDim)
    traj      = [[list(a) for a in coord]]
    E_hist    = [Ene]
    Elj_hist  = [ELJ]
    Ecoul_hist = [ECoul]
    acc_hist  = [1.0]
    move_hist = ["init"]

    Accepted = 0
    frac_simple_move = 1 - frac_swap
    kT = cstboltz * Temperature

    print("step %6d  E= %.3f" % (0, Ene))

    for step in range(1, nsteps+1):
        ra = randint(0, len(coord)-1)

        if random() < frac_simple_move:
            # --- displacement of one coordinate of atom ra ---
            movetype = "move"
            rc = randint(0, 1)
            old = coord[ra][rc]
            factor = (2*random() - 1) * deltaRmax
            z = old + factor
            halfbox = BoxDim[rc]/2
            z = z - SignR(halfbox, z) - SignR(halfbox, z - BoxDim[rc])   # wrap into the box
            coord[ra][rc] = z

            En, LJn, Con = Calc_Ene2(coord, Epsilon, Rmin, Dielec, CutOffSquare, BoxDim)
            dE = En - Ene
            if dE < 0.0 or random() < exp(-dE/kT):
                Ene, ELJ, ECoul = En, LJn, Con
                Accepted += 1
            else:
                coord[ra][rc] = old                     # reject -> undo

        else:
            # --- charge swap: exchange the positions of atoms ra and ra2 ---
            movetype = "swap"
            ra2 = randint(0, len(coord)-1)
            oxa, oya = coord[ra][0],  coord[ra][1]
            oxb, oyb = coord[ra2][0], coord[ra2][1]
            coord[ra][0],  coord[ra][1]  = oxb, oyb
            coord[ra2][0], coord[ra2][1] = oxa, oya

            En, LJn, Con = Calc_Ene2(coord, Epsilon, Rmin, Dielec, CutOffSquare, BoxDim)
            dE = En - Ene
            if dE < 0.0 or random() < exp(-dE/kT):
                Ene, ELJ, ECoul = En, LJn, Con
                Accepted += 1
            else:
                coord[ra][0],  coord[ra][1]  = oxa, oya  # reject -> undo
                coord[ra2][0], coord[ra2][1] = oxb, oyb

        traj.append([list(a) for a in coord])
        E_hist.append(Ene)
        Elj_hist.append(ELJ)
        Ecoul_hist.append(ECoul)
        acc_hist.append(Accepted/step)
        move_hist.append(movetype)

        if step % 1000 == 0:
            print("step %6d  E= %.3f  P(accept)= %.3f" % (step, Ene, Accepted/step))

    print("\nFinal acceptance ratio: %.3f" % (Accepted/nsteps))
    return traj, E_hist, Elj_hist, Ecoul_hist, acc_hist, move_hist

## 5. Parameters

These are the same parameters exposed by the sliders/entry boxes of the original GUI.
Change any of them and re-run **this cell together with the Initialisation and Run cells just
below** to explore their effect (or use *Kernel → Restart & Run All*). Values are in the
toy model's arbitrary units.

> **Note.** The **system** parameters below are the same shared set used by the
> energy-minimization and molecular-dynamics notebooks (`nAtoms=20`, `Radius=25`,
> `Epsilon=25`, `qat=Radius`, `Seed=100`, …), so all methods act on the identical starting
> configuration. Monte Carlo adds its own **sampling** controls (`deltaRmax`, `frac_swap`,
> `Temperature`).

### System and its properties

| Parameter | Meaning | Typical value / range |
|---|---|---|
| `nAtoms` | number of particles | 2–40 |
| `Radius` | particle radius (drawn size, and default absolute charge) | 10–40 — must leave room to place all atoms |
| `Rmin` | position of the LJ energy minimum | `2.24 * Radius` |
| `BoxDim` | box dimensions (periodic) | `[500, 500]` |
| `Epsilon` | LJ well depth | 1–100 |
| `Dielec` | dielectric constant (charge screening) | 1 (vacuum) – 80 (water) |
| `qat` | absolute charge per atom | defaults to `Radius` |
| `frac_neg` | fraction of negative charges | 0–1 |
| `CutOff` | non-bonded cutoff distance | 250 |

### Monte Carlo controls

| Parameter | Meaning | Typical value / range |
|---|---|---|
| `deltaRmax` | maximum displacement of a trial move (tuned for ~0.35 acceptance) | 30 |
| `frac_swap` | fraction of trial moves that are charge swaps | 0–1 (0.2) |
| `Temperature` | temperature `T` in the Boltzmann factor (higher → more uphill moves accepted) | 300 |
| `cstboltz` | Boltzmann constant (sets the energy scale of $k_B T$) | fixed |
| `Seed` | random-number seed (reproducibility) | 100 |
| `nsteps` | number of Monte Carlo trial moves | 20000 |

In [ ]:
nAtoms  = 20              # number of particles
Radius  = 25.0            # particle radius (must leave room to place all atoms)
Rmin    = 2.24 * Radius   # distance at which the LJ energy is minimal
BoxDim  = [500, 500]      # box dimensions
Epsilon = 25.0            # LJ well depth
Dielec  = 1.0             # dielectric constant
qat     = Radius          # atom absolute charge
frac_neg = 0.5            # fraction of negative charges
OverlapFr = 0.0           # fraction of overlap allowed when placing atoms
CutOff  = 250             # non-bonded cutoff
CutOffSquare = CutOff**2

# --- Monte Carlo controls ---
deltaRmax   = 30.0             # maximum displacement of a trial move (tuned: acceptance ~0.35)
frac_swap   = 0.2              # fraction of trial moves that are charge swaps
Temperature = 300.0           # temperature in the Boltzmann acceptance factor
cstboltz    = 8.3502e-03       # Boltzmann constant (J/mol/K) -> sets the k_B T energy scale
Seed        = 100             # random number seed (reproducibility, shared with the other notebooks)
nsteps      = 20000           # number of Monte Carlo trial moves

## 6. Initialisation

Generate random, non-overlapping starting positions and assign charges
(a fraction `frac_neg` negative, the rest positive). `seed(Seed)` makes the run
reproducible.

In [ ]:
import sys

### generate random, non-overlapping coordinates ###
def InitConf(n, dim, radius, qat, frac_neg):
    seed(Seed)
    print("Initializing box, please wait...")
    tmp_coord = []
    i = 0
    ntrial = 0
    nneg = int(float(n) * frac_neg)
    npos = n - nneg

    # first atom (no overlap check)
    x = random()*(dim[0]-radius) + radius
    y = random()*(dim[1]-radius) + radius
    charge = -qat
    if npos == n:
        charge = qat
    i += 1
    tmp_coord.append([x, y, charge])

    # remaining negative charges
    while i < nneg:
        x = random()*(dim[0]-radius) + radius
        y = random()*(dim[1]-radius) + radius
        OVERLAP = 1
        for j in range(i):
            if dist(tmp_coord[j], [x, y]) < (1-OverlapFr)*2*radius:
                OVERLAP = 0
        if OVERLAP:
            tmp_coord.append([x, y, -qat])
            i += 1
        ntrial += 1
        if ntrial > 100000:
            print("initialisation failed -> reduce radius or number of atoms")
            sys.exit()

    # remaining positive charges
    while i < n:
        x = random()*(dim[0]-radius) + radius
        y = random()*(dim[1]-radius) + radius
        OVERLAP = 1
        for j in range(i):
            if dist(tmp_coord[j], [x, y]) < (1-OverlapFr)*2*radius:
                OVERLAP = 0
        if OVERLAP:
            tmp_coord.append([x, y, qat])
            i += 1
        ntrial += 1
        if ntrial > 100000:
            print("initialisation failed -> reduce radius or number of atoms")
            sys.exit()
    return tmp_coord


Atom_Coord = InitConf(nAtoms, BoxDim, Radius, qat, frac_neg)
Color = [charge_color(a[2], qat) for a in Atom_Coord]
print(f"Placed {len(Atom_Coord)} atoms.")

## 7. Run the Monte Carlo

Run `nsteps` Metropolis trial moves starting from the initial configuration, recording the
energy, the running acceptance ratio and a snapshot at every step.

In [ ]:
traj, E_hist, Elj_hist, Ecoul_hist, acc_hist, move_hist = run_mc(Atom_Coord)
print(f"Done: {len(traj)-1} MC steps. Final E = {E_hist[-1]:.2f}, "
      f"E range sampled = [{min(E_hist):.1f}, {max(E_hist):.1f}]")

## 8. Energy and acceptance

The left panel shows the total, Lennard-Jones and Coulomb energies along the Monte Carlo run:
the energy drops quickly at first (the random start relaxes) and then **fluctuates around an
equilibrium average** — Monte Carlo samples a *thermal ensemble*, it does not drive the energy
to a single minimum. The right panel shows the running **acceptance ratio**; `deltaRmax` and
`Temperature` are the knobs that tune it (rule of thumb: aim for roughly 30–50 %).

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

steps = range(len(E_hist))
ax1.plot(steps, E_hist,     label="E$_{tot}$", lw=1.5)
ax1.plot(steps, Elj_hist,   label="E$_{LJ}$",  lw=1.0)
ax1.plot(steps, Ecoul_hist, label="E$_{Coul}$", lw=1.0)
ax1.set_xlabel("MC step")
ax1.set_ylabel("energy")
ax1.set_title("Energy along the Monte Carlo run")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(steps, acc_hist, color="tab:green", lw=1.2)
ax2.set_xlabel("MC step")
ax2.set_ylabel("acceptance ratio")
ax2.set_ylim(0, 1)
ax2.set_title(f"Running acceptance (final = {acc_hist[-1]:.2f})")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Visualise the system

Start and end configurations side by side. White = positive charge, dark = negative.

In [ ]:
def draw_config(ax, coord, title):
    ax.set_xlim(0, BoxDim[0])
    ax.set_ylim(0, BoxDim[1])
    ax.set_aspect('equal')
    ax.set_facecolor("#ccddff")
    ax.set_title(title)
    ax.invert_yaxis()   # match the original canvas (y downwards)
    for a in coord:
        col = charge_color(a[2], qat)
        ax.add_patch(Circle((a[0], a[1]), Radius, facecolor=col, edgecolor="black", lw=0.8))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5.5))
draw_config(ax1, traj[0],  f"Initial  (E = {E_hist[0]:.1f})")
draw_config(ax2, traj[-1], f"Final  (E = {E_hist[-1]:.1f})")
plt.tight_layout()
plt.show()

## 10. Animation of the Monte Carlo run

Replays the whole trajectory. This reproduces the live view of the original GUI.
(To keep it light, only every few frames are shown — adjust `stride`.)

In [ ]:
stride = max(1, len(traj)//120)   # cap at ~120 frames
frames = list(range(0, len(traj), stride))

fig, ax = plt.subplots(figsize=(6, 6))
ax.set_xlim(0, BoxDim[0])
ax.set_ylim(0, BoxDim[1])
ax.set_aspect('equal')
ax.set_facecolor("#ccddff")
ax.invert_yaxis()

circles = [Circle((a[0], a[1]), Radius,
                  facecolor=charge_color(a[2], qat), edgecolor="black", lw=0.8)
           for a in traj[0]]
for c in circles:
    ax.add_patch(c)
title = ax.set_title("")

def update(frame_idx):
    f = frames[frame_idx]
    for c, a in zip(circles, traj[f]):
        c.center = (a[0], a[1])
    title.set_text(f"step {f}   E = {E_hist[f]:.1f}   ({move_hist[f]})")
    return circles + [title]

anim = FuncAnimation(fig, update, frames=len(frames), interval=80, blit=False)
plt.close(fig)   # avoid a duplicate static figure
anim

## 11. Monte Carlo vs energy minimization and molecular dynamics

All the notebooks in this series act on the **same system** (20 particles, identical
parameters and random seed `Seed = 100`) but explore it in fundamentally different ways:

* **Energy minimization** (`LJ-ELEC_EM-steepest`, `-conjugate`, `-simplex`) only moves
  *downhill* and stops at the nearest local minimum — it answers *"what stable arrangement is
  closest to the start?"*.
* **Molecular dynamics** (`LJ-ELEC_MD-Verlet`) integrates Newton's equations at constant
  energy, giving a physical time trajectory whose temperature is set by the initial velocities.
* **Monte Carlo** (this notebook) makes *random* trial moves accepted by the Metropolis rule,
  and so samples the **Boltzmann distribution** at temperature `T`. It does not follow a
  physical trajectory in time, but it is often the most efficient way to sample equilibrium
  averages and — because it accepts occasional uphill moves — to escape local minima. The
  extra **charge-swap** move lets it rearrange the charge pattern non-locally, something the
  purely local EM/MD moves cannot do easily.

Two useful limits to try: setting `Temperature` very **low** turns Monte Carlo into a stochastic
*minimizer* (almost only downhill moves accepted); raising it lets the system roam freely.
Slowly lowering `T` during a run (*simulated annealing*) is a classic way to search for deep
minima.